# PathSnap: Exploratory Data Analysis & Geospatial Verification
**UE24CS352A Machine Learning | Map-Matching & Route Reconstruction**

This notebook provides a structured, reproducible exploratory analysis of the PathSnap dataset covering:
1. Dataset Inventory & Partition Verification (80 Train + 10 Student Test trips)
2. Coordinate Validity & Geodesic Distance Justification
3. Temporal Sampling Rate (1 Hz Verification) & Anomaly Gap Detection
4. Road Network Graph Topology & Connectivity
5. Interactive Folium Geographical Visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import IPython.display as display

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## 1. Dataset Inventory and Partition Breakdown
We inspect the inventory of all 90 permitted trips generated by our data loader.

In [ ]:
inventory_path = Path('../outputs/eda/dataset_inventory.csv')
if not inventory_path.exists():
    inventory_path = Path('outputs/eda/dataset_inventory.csv')

inventory = pd.read_csv(inventory_path)
print(f"Total trips analysed: {len(inventory)}")
print(f"Partition counts:\n{inventory['partition'].value_counts()}\n")
print(f"Total GPS observations across all trips: {inventory['gps_observations'].sum():,}")
print(f"Invalid coordinates found: {inventory['invalid_coordinates'].sum()}")
print(f"Missing coordinates found: {inventory['missing_coordinates'].sum()}")
print(f"Duplicate timestamps found: {inventory['duplicate_observations'].sum()}")

inventory[['trip_id', 'partition', 'gps_observations', 'nodes', 'arcs', 'trajectory_length_m', 'duration_s']].head(10)

## 2. Temporal Sampling Frequency & Anomaly Analysis
The problem specification states GPS sampling is nominally 1 Hz (~97% at 1.0 s). We verify this empirically across all observations.

In [ ]:
print(f"Median sampling interval across all trips: {inventory['median_interval_s'].median():.2f} s")
print(f"Trips with GPS gaps > 5 seconds: {(inventory['gps_gaps_over_5s'] > 0).sum()} of {len(inventory)}")

anomalies = inventory.sort_values('max_interval_s', ascending=False)[['trip_id', 'partition', 'max_interval_s', 'gps_gaps_over_5s', 'duration_s']].head(5)
print("Top 5 Trips with Largest GPS Gaps (e.g. tunnels / signal dropouts):")
display.display(anomalies)

In [ ]:
# Display the generated sampling interval and duration plots
plots_dir = Path('../outputs/eda/plots') if Path('../outputs/eda/plots').exists() else Path('outputs/eda/plots')

for plot_name in ['median_interval_s.png', 'duration_s.png']:
    p = plots_dir / plot_name
    if p.exists():
        print(f"Displaying {plot_name}:")
        display.display(display.Image(filename=str(p)))

## 3. Spatial Trajectory Extent & Geodesic Distance
We verify trajectory lengths in metres using geodesic (Haversine) formulas, rejecting raw Euclidean degree calculations.

In [ ]:
print("Trajectory Length (metres) Summary Statistics:")
print(inventory['trajectory_length_m'].describe())

for plot_name in ['gps_trajectory.png', 'lon_lat_distribution.png', 'trajectory_length_m.png']:
    p = plots_dir / plot_name
    if p.exists():
        display.display(display.Image(filename=str(p)))

## 4. Road Network Size and Graph Connectivity
Network topology summary: number of nodes, directed arcs, and connected components per trip.

In [ ]:
print("Road Network Node and Arc Counts:")
print(inventory[['nodes', 'arcs', 'connected_components']].describe())

p = plots_dir / 'connected_components.png'
if p.exists():
    display.display(display.Image(filename=str(p)))

## 5. Interactive Folium Geographical Visualization
Visualizing raw noisy GPS points (blue), road network geometry (gray), and ground-truth route (green) with layer controls.

In [ ]:
map_path = Path('../outputs/maps/00000001_eda_map.html') if Path('../outputs/maps/00000001_eda_map.html').exists() else Path('outputs/maps/00000001_eda_map.html')
if map_path.exists():
    print(f"Folium interactive HTML map verified ({map_path.stat().st_size / (1024*1024):.1f} MB)")
    print("Map features:")
    print(" - Blue markers: Noisy raw GPS observations")
    print(" - Gray polylines: Road network segments from .nodes and .arcs")
    print(" - Green polyline: Reference vehicle path from .route")
    print(" - Interactive LayerControl enabled for evaluator inspection.")